[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/16_cross_entropy.ipynb)

# 🟢 Easy: Cross-Entropy Loss

Implement **cross-entropy loss** from scratch.

$$\text{CE}(x, y) = -\log\frac{e^{x_y}}{\sum_j e^{x_j}}$$

### Signature
```python
def cross_entropy_loss(logits: Tensor, targets: Tensor) -> Tensor:
    # logits: (B, C) float, targets: (B,) long indices
    # Returns: scalar loss (mean over batch)
```

### Rules
- Do NOT use `F.cross_entropy` or `nn.CrossEntropyLoss`
- Must be numerically stable (use logsumexp trick)

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.5 MB/s eta 0:00:00


In [2]:
import torch

In [3]:
# ✏️ YOUR IMPLEMENTATION HERE

def cross_entropy_loss(logits, targets):
    log_probs = logits - torch.logsumexp(logits, dim=-1, keepdim=True)
    return -log_probs[torch.arange(targets.shape[0]), targets].mean()

In [4]:
# 🧪 Debug
logits = torch.randn(4, 10)
targets = torch.randint(0, 10, (4,))
print('Loss:', cross_entropy_loss(logits, targets))
print('Ref: ', torch.nn.functional.cross_entropy(logits, targets))

Loss: tensor(2.6648)
Ref:  tensor(2.6648)


In [5]:
# ✅ SUBMIT
from torch_judge import check
check('cross_entropy')


🧪 Testing: Cross-Entropy Loss (Easy)
──────────────────────────────────────────────────
  ✅ [1/4] Matches F.cross_entropy (13.6ms)
  ✅ [2/4] Numerical stability (0.9ms)
  ✅ [3/4] Scalar output (0.4ms)
  ✅ [4/4] Gradient flow (16.2ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (31.1ms total)
  Progress saved. Run status() to see your dashboard.



In [6]:
logits = torch.tensor([
    [1., 2., 3.],
    [2., 1., 0.]
])

targets = torch.tensor([2, 1])

In [8]:
print(logits.shape)
print(targets.shape)

torch.Size([2, 3])
torch.Size([2])


In [9]:
lse = torch.logsumexp(
    logits, dim=-1, keepdim=True
)

print(lse)

tensor([[3.4076],
        [2.4076]])


In [10]:
log_probs = logits - lse
print(log_probs)

tensor([[-2.4076, -1.4076, -0.4076],
        [-0.4076, -1.4076, -2.4076]])


In [11]:
torch.arange(targets.shape[0])
# torch.arange(2)

tensor([0, 1])

In [16]:
target_log_probs = log_probs[
    torch.arange(targets.shape[0]),
    targets
]

print(target_log_probs)
print([
    log_probs[0,2],
    log_probs[1,1]
    ])
print(log_probs[[0, 1], [2, 1]])

tensor([-0.4076, -1.4076])
[tensor(-0.4076), tensor(-1.4076)]
tensor([-0.4076, -1.4076])


In [13]:
losses = -target_log_probs

print(losses)

print(losses.mean())

tensor([0.4076, 1.4076])
tensor(0.9076)


In [14]:
log_probs[:, targets]

tensor([[-0.4076, -1.4076],
        [-2.4076, -1.4076]])

# 더 이해하기 쉬운 버전으로 Cross-Entropy Loss 구현

$$
\boxed{
L_{\mathrm{CE}}
= -\frac{1}{B}\sum_{i=1}^{B}
\left[
z_{i,y_i} - \operatorname{LogSumExp}(\mathbf{z}_i)
\right]
= -\frac{1}{B}\sum_{i=1}^{B}
\log\left(
\frac{e^{z_{i,y_i}}}{\sum_{j=1}^{C}e^{z_{i,j}}}
\right)
}
$$

In [17]:
import torch

def cross_entropy_loss(
    logits: torch.Tensor,  # (B, C)
    targets: torch.Tensor  # (B,)
) -> torch.Tensor:

    # 1. LogSumExp for each sample
    lse = torch.logsumexp(logits, dim=-1, keepdim=True)  # (B, 1)

    # 2. Log-Softmax
    log_probs = logits - lse  # (B, C)

    # 3. Select the correct class for each sample
    batch_indices = torch.arange(logits.shape[0], device=logits.device)
    target_log_probs = log_probs[batch_indices, targets]  # (B,)

    # 4. Negative Log-Likelihood (NLL)
    losses = -target_log_probs  # (B,)

    # 5. Mean Cross-Entropy Loss
    return losses.mean()  # Scalar ()


In [18]:

# Debugging: Inspect each intermediate tensor

logits = torch.tensor([
    [1., 2., 3.],
    [2., 1., 0.]
])
targets = torch.tensor([2, 1])

lse = torch.logsumexp(logits, dim=-1, keepdim=True)
log_probs = logits - lse

batch_indices = torch.arange(logits.shape[0])
target_log_probs = log_probs[batch_indices, targets]

losses = -target_log_probs
final_loss = losses.mean()

print("[1] Logits")
print(logits)
print("Shape:", logits.shape)

print("\n[2] LogSumExp")
print(lse)
print("Shape:", lse.shape)

print("\n[3] Log-Probabilities")
print(log_probs)
print("Shape:", log_probs.shape)

print("\n[4] Selected Target Log-Probabilities")
print(target_log_probs)
print("Shape:", target_log_probs.shape)

print("\n[5] Individual NLL")
print(losses)
print("Shape:", losses.shape)

print("\n[6] Final Cross-Entropy Loss")
print(final_loss)
print("Shape:", final_loss.shape)


[1] Logits
tensor([[1., 2., 3.],
        [2., 1., 0.]])
Shape: torch.Size([2, 3])

[2] LogSumExp
tensor([[3.4076],
        [2.4076]])
Shape: torch.Size([2, 1])

[3] Log-Probabilities
tensor([[-2.4076, -1.4076, -0.4076],
        [-0.4076, -1.4076, -2.4076]])
Shape: torch.Size([2, 3])

[4] Selected Target Log-Probabilities
tensor([-0.4076, -1.4076])
Shape: torch.Size([2])

[5] Individual NLL
tensor([0.4076, 1.4076])
Shape: torch.Size([2])

[6] Final Cross-Entropy Loss
tensor(0.9076)
Shape: torch.Size([])


In [19]:

import torch
import torch.nn.functional as F

test_cases = [
    ("Basic", [[1., 2., 3.], [2., 1., 0.]], [2, 1]),
    ("Uniform logits", [[0., 0., 0.]], [1]),
    ("Correct prediction", [[0., 0., 10.]], [2]),
    ("Wrong prediction", [[0., 0., 10.]], [0]),
    ("Numerical stability", [[1000., 1001., 1002.]], [2]),
]

for name, x, y in test_cases:
    logits = torch.tensor(x)
    targets = torch.tensor(y)

    my_loss = cross_entropy_loss(logits, targets)
    ref_loss = F.cross_entropy(logits, targets)

    print(f"\n[{name}]")
    print(f"My Loss:   {my_loss.item():.6f}")
    print(f"Reference: {ref_loss.item():.6f}")
    print(f"Allclose:  {torch.allclose(my_loss, ref_loss)}")
    print(f"Shape:     {my_loss.shape}")



[Basic]
My Loss:   0.907606
Reference: 0.907606
Allclose:  True
Shape:     torch.Size([])

[Uniform logits]
My Loss:   1.098612
Reference: 1.098612
Allclose:  True
Shape:     torch.Size([])

[Correct prediction]
My Loss:   0.000091
Reference: 0.000091
Allclose:  False
Shape:     torch.Size([])

[Wrong prediction]
My Loss:   10.000091
Reference: 10.000091
Allclose:  True
Shape:     torch.Size([])

[Numerical stability]
My Loss:   0.407593
Reference: 0.407606
Allclose:  False
Shape:     torch.Size([])


In [20]:
logits = torch.tensor(
    [[1., 2., 3.]],
    requires_grad=True
)
targets = torch.tensor([2])

loss = cross_entropy_loss(logits, targets)
loss.backward()

print("Loss:", loss)
print("Gradient:", logits.grad)

Loss: tensor(0.4076, grad_fn=<MeanBackward0>)
Gradient: tensor([[ 0.0900,  0.2447, -0.3348]])


In [21]:

import torch
import torch.nn.functional as F

test_cases = [
    ("Basic", [[1., 2., 3.]], [2]),
    ("Uniform", [[0., 0., 0.]], [1]),
    ("Correct", [[0., 0., 10.]], [2]),
    ("Wrong", [[0., 0., 10.]], [0]),
    ("Large logits", [[1000., 1001., 1002.]], [2]),
    ("Extreme wrong", [[0., 0., 1000.]], [0]),
]

for name, x, y in test_cases:
    logits = torch.tensor(x)
    targets = torch.tensor(y)

    idx = torch.arange(logits.shape[0])

    # A. Original logsumexp implementation
    loss_a = (
        torch.logsumexp(logits, dim=-1)
        - logits[idx, targets]
    ).mean()

    # B. Max-shifted stable implementation
    shifted = logits - logits.max(
        dim=-1, keepdim=True
    ).values

    loss_b = (
        torch.log(torch.exp(shifted).sum(dim=-1))
        - shifted[idx, targets]
    ).mean()

    # C. Softmax followed by -log
    probs = torch.softmax(logits, dim=-1)
    loss_c = -torch.log(probs[idx, targets]).mean()

    # Reference
    ref = F.cross_entropy(logits, targets)

    print(f"\n[{name}]")
    for label, loss in [
        ("A. LogSumExp", loss_a),
        ("B. Max-shifted", loss_b),
        ("C. Softmax+Log", loss_c),
    ]:
        print(
            f"{label:17s}: {loss.item():.8f} "
            f"| allclose={torch.allclose(loss, ref)}"
        )

    print(f"Reference        : {ref.item():.8f}")



[Basic]
A. LogSumExp     : 0.40760589 | allclose=True
B. Max-shifted   : 0.40760598 | allclose=True
C. Softmax+Log   : 0.40760598 | allclose=True
Reference        : 0.40760595

[Uniform]
A. LogSumExp     : 1.09861231 | allclose=True
B. Max-shifted   : 1.09861231 | allclose=True
C. Softmax+Log   : 1.09861231 | allclose=True
Reference        : 1.09861231

[Correct]
A. LogSumExp     : 0.00009060 | allclose=False
B. Max-shifted   : 0.00009083 | allclose=True
C. Softmax+Log   : 0.00009084 | allclose=True
Reference        : 0.00009083

[Wrong]
A. LogSumExp     : 10.00009060 | allclose=True
B. Max-shifted   : 10.00009060 | allclose=True
C. Softmax+Log   : 10.00009060 | allclose=True
Reference        : 10.00009060

[Large logits]
A. LogSumExp     : 0.40759277 | allclose=False
B. Max-shifted   : 0.40760598 | allclose=True
C. Softmax+Log   : 0.40760598 | allclose=True
Reference        : 0.40760595

[Extreme wrong]
A. LogSumExp     : 1000.00000000 | allclose=True
B. Max-shifted   : 1000.00000000

# 학습용 최종 코드

In [22]:

import torch

def cross_entropy_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    # logits: (B, C), targets: (B,)

    # 1. Numerical stability
    x_max = logits.max(dim=-1, keepdim=True).values
    shifted = logits - x_max                # (B, C)

    # 2. LogSumExp (after max subtraction)
    log_sum_exp = torch.log(
        torch.exp(shifted).sum(dim=-1)
    )                                       # (B,)

    # 3. Select target-class logits
    batch_indices = torch.arange(logits.shape[0], device=logits.device)
    target_logits = shifted[batch_indices, targets]  # (B,)

    # 4. Mean Cross-Entropy Loss
    return (log_sum_exp - target_logits).mean()       # ()
